Cell 1 — Imports

In [1]:
import os
import time
import numpy as np
import pandas as pd
import faiss

from typing import Dict, List, Tuple
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import normalize

Cell 2 — Config

In [2]:
emb_folder = "\Embeddings"
EMBEDDING_FILES = {
    "bert_finetuned": emb_folder + "\\bert_finetuned_embeddings.xlsx",
    "gemini": emb_folder + "\\Gemini_Embedding.xlsx",
    "qwen3_8b": emb_folder + "\\Qwen3_Embedding_8B.xlsx",
    "sbert": emb_folder + "\\SBERT_Embedding_2_classification.xlsx",
}


SIM_METRICS = ["cosine", "dot", "neg_l2"]
TOPK_LIST = [1, 5, 10, 20]

# RRF
RRF_K = 60
FUSION_DEPTH = 100

# Soft filtering params
CLASS_TOPK = 3
BETA = 1.0
N_SPLITS = 5
RANDOM_STATE = 42

# IVF params
IVF_NLIST = 128
IVF_NPROBE = 10

# PQ params
PQ_NLIST = 128
PQ_NBITS = 8
PQ_NPROBE = 10

# HNSW params
HNSW_M = 32
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH = 64

# Include oracle filtered-static run in fusion?
INCLUDE_FILTERED_STATIC = True

HYBRID_METHOD_NAME = "hybrid_rrf_no_flat"

# Flat is intentionally kept only as an exact-search baseline and is NOT used in hybrid fusion.
EXCLUDE_FLAT_FROM_HYBRID = True

OUT_XLSX = "hybrid_rrf_no_flat_summary.xlsx"
OUT_CSV = "hybrid_rrf_no_flat_run.csv"


Cell 3 — Load Embedding Files

In [3]:
def load_embedding_xlsx(path):

    df = pd.read_excel(path, engine="openpyxl")

    # label column
    label_candidates = [c for c in df.columns if str(c).lower() in ("label","y","class")]
    if not label_candidates:
        raise ValueError(f"{path} → label column not found")
    label_col = label_candidates[0]

    # id column
    id_candidates = [c for c in df.columns if str(c).lower() in ("filename","file","text_file","id","file_id")]
    if id_candidates:
        id_col = id_candidates[0]
    else:
        non_num = [c for c in df.columns if c != label_col]
        id_col = non_num[0]

    # embedding columns
    emb_cols = [
        c for c in df.columns
        if c not in (label_col, id_col)
        and (
            str(c).isdigit()
            or str(c).lower().startswith("e")
            or str(c).lower().startswith("emb")
        )
    ]

    if len(emb_cols) == 0:
        raise ValueError(f"{path} → embedding columns not found")

    X = df[emb_cols].to_numpy(dtype=np.float32)
    y = df[label_col].to_numpy()
    ids = df[id_col].astype(str).to_numpy()

    print(f"{path} → embeddings shape:", X.shape)

    return df, X, y, ids

Cell 4 — Evaluation Metrics

In [4]:
def precision_at_k(rels, k):
    return float(np.sum(rels[:k])) / float(k)


def recall_at_k(rels, k, total_rel):
    if total_rel == 0:
        return 0.0
    return float(np.sum(rels[:k])) / float(total_rel)


def dcg_at_k(rels, k):
    rels = rels[:k]
    denom = np.log2(np.arange(2, len(rels) + 2))
    return float(np.sum(rels / denom))


def ndcg_at_k(rels, k):
    dcg = dcg_at_k(rels, k)
    ideal = np.sort(rels)[::-1]
    idcg = dcg_at_k(ideal, k)
    if idcg == 0:
        return 0.0
    return dcg / idcg


def mrr_at_k(rels, k):
    rels = rels[:k]
    idx = np.where(rels == 1)[0]
    if len(idx) == 0:
        return 0.0
    return 1.0 / float(idx[0] + 1)

Cell 5 — Similarity Helpers

In [5]:
def prepare_vectors(X, metric):
    if metric == "cosine":
        return normalize(X, axis=1).astype(np.float32)
    return np.asarray(X, dtype=np.float32)


def scores_for_metric(q, X, metric):
    """
    Return scores where larger is better.
    cosine / dot : higher better
    neg_l2       : negative squared L2, so higher is better
    """
    if metric in ("cosine", "dot"):
        return X @ q

    if metric == "neg_l2":
        x2 = np.sum(X * X, axis=1)
        q2 = float(np.sum(q * q))
        return -(x2 + q2 - 2.0 * (X @ q))

    raise ValueError(f"Unknown metric: {metric}")

Cell 6 — Soft Classification (for classifier-guided retrieval)

In [6]:
def cv_soft_probabilities(X, y, n_splits=5, random_state=42):

    classes = np.unique(y)
    class_to_index = {c: i for i, c in enumerate(classes)}

    N = len(y)
    C = len(classes)

    P = np.zeros((N, C), dtype=np.float32)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for train_idx, test_idx in skf.split(X, y):

        Xtr, Xte = X[train_idx], X[test_idx]
        ytr = y[train_idx]

        clf = LogisticRegression(max_iter=2000)

        clf.fit(Xtr, ytr)

        proba = clf.predict_proba(Xte)

        for j, c in enumerate(clf.classes_):
            P[test_idx, class_to_index[c]] = proba[:, j]

    return P, classes, {"class_to_index": class_to_index}

Cell 7 — Utility: convert neighbor indices to ranked-list rows

In [7]:
def build_run_rows_from_neighbors(method_name, metric, query_idx, nbrs, ids, y, depth):
    rows = []
    rank = 1
    for j in nbrs[:depth]:
        if j < 0:
            continue
        rows.append({
            "method": method_name,
            "metric": metric,
            "query_id": ids[query_idx],
            "query_label": int(y[query_idx]),
            "doc_id": ids[j],
            "doc_label": int(y[j]),
            "rank": rank,
            "relevance": int(y[j] == y[query_idx]),
        })
        rank += 1
    return rows

Cell 8 — FAISS Flat ranked-list builder 

This cell is kept for baseline experiments only. Flat search is not included in the updated hybrid RRF fusion.


In [8]:
def build_run_flat(X, y, ids, metric, depth):
    X_use = prepare_vectors(X, metric)
    N, d = X_use.shape

    if metric == "neg_l2":
        index = faiss.IndexFlatL2(d)
    else:
        index = faiss.IndexFlatIP(d)

    index.add(X_use)

    search_k = min(N, depth + 1)
    D, I = index.search(X_use, search_k)

    rows = []
    for i in range(N):
        nbrs = I[i]
        nbrs = nbrs[(nbrs >= 0) & (nbrs != i)]  # remove self
        rows.extend(build_run_rows_from_neighbors("flat", metric, i, nbrs, ids, y, depth))

    return pd.DataFrame(rows)

Cell 9 — IVF ranked-list builder

In [9]:
def build_run_ivf(X, y, ids, metric, depth, nlist=128, nprobe=10):
    X_use = prepare_vectors(X, metric)
    N, d = X_use.shape

    if metric == "neg_l2":
        quantizer = faiss.IndexFlatL2(d)
        index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)
    else:
        quantizer = faiss.IndexFlatIP(d)
        index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)

    if not index.is_trained:
        index.train(X_use)

    index.add(X_use)
    index.nprobe = nprobe

    search_k = min(N, depth + 1)
    D, I = index.search(X_use, search_k)

    rows = []
    for i in range(N):
        nbrs = I[i]
        nbrs = nbrs[(nbrs >= 0) & (nbrs != i)]  # remove self
        rows.extend(build_run_rows_from_neighbors("ivf", metric, i, nbrs, ids, y, depth))

    return pd.DataFrame(rows)

Cell 10 — IVFPQ ranked-list builder

In [10]:
def choose_pq_m(d):
    # Prefer sub-dim 8, then 16, then 4
    for sd in [8, 16, 4]:
        if d % sd == 0:
            return d // sd
    for M in range(min(96, d), 1, -1):
        if d % M == 0:
            return M
    raise ValueError(f"Cannot find suitable PQ_M for d={d}")


def build_run_ivfpq(X, y, ids, metric, depth, nlist=128, nprobe=10, nbits=8):
    X_use = prepare_vectors(X, metric)
    N, d = X_use.shape
    M = choose_pq_m(d)

    if metric == "neg_l2":
        quantizer = faiss.IndexFlatL2(d)
        index = faiss.IndexIVFPQ(quantizer, d, nlist, M, nbits, faiss.METRIC_L2)
    else:
        quantizer = faiss.IndexFlatIP(d)
        index = faiss.IndexIVFPQ(quantizer, d, nlist, M, nbits, faiss.METRIC_INNER_PRODUCT)

    if not index.is_trained:
        index.train(X_use)

    index.add(X_use)
    index.nprobe = nprobe

    search_k = min(N, depth + 1)
    D, I = index.search(X_use, search_k)

    rows = []
    for i in range(N):
        nbrs = I[i]
        nbrs = nbrs[(nbrs >= 0) & (nbrs != i)]  # remove self
        rows.extend(build_run_rows_from_neighbors("ivfpq", metric, i, nbrs, ids, y, depth))

    return pd.DataFrame(rows)

Cell 11 — HNSW ranked-list builder

In [11]:
def build_run_hnsw(X, y, ids, metric, depth, M=32, efc=200, efs=64):
    X_use = prepare_vectors(X, metric)
    N, d = X_use.shape

    if metric == "neg_l2":
        index = faiss.IndexHNSWFlat(d, M, faiss.METRIC_L2)
    else:
        index = faiss.IndexHNSWFlat(d, M, faiss.METRIC_INNER_PRODUCT)

    index.hnsw.efConstruction = int(efc)
    index.hnsw.efSearch = int(efs)

    index.add(X_use)

    search_k = min(N, depth + 1)
    D, I = index.search(X_use, search_k)

    rows = []
    for i in range(N):
        nbrs = I[i]
        nbrs = nbrs[(nbrs >= 0) & (nbrs != i)]  # remove self
        rows.extend(build_run_rows_from_neighbors("hnsw", metric, i, nbrs, ids, y, depth))

    return pd.DataFrame(rows)

Cell 12 — Filtered Static ranked-list builder

In [12]:
def build_run_filtered_static(X, y, ids, metric, depth):
    X_use = prepare_vectors(X, metric)
    rows = []

    classes = np.unique(y)
    class_to_idx = {c: np.where(y == c)[0] for c in classes}

    for i in range(len(ids)):
        cand = class_to_idx[y[i]]
        cand = cand[cand != i]  # remove self

        if cand.size == 0:
            continue

        q = X_use[i]
        Xc = X_use[cand]

        scores = scores_for_metric(q, Xc, metric)

        take = min(depth, len(cand))
        top_local = np.argpartition(-scores, take - 1)[:take]
        top_local = top_local[np.argsort(-scores[top_local])]
        nbrs = cand[top_local]

        rows.extend(build_run_rows_from_neighbors("filtered_static", metric, i, nbrs, ids, y, depth))

    return pd.DataFrame(rows)

Cell 13 — Classifier-guided / Soft Filtering ranked-list builder

In [13]:
def build_run_soft_filtering(
    X, y, ids, metric, depth,
    class_topk=3, beta=1.0,
    n_splits=5, random_state=42
):
    P, classes, meta = cv_soft_probabilities(
        X, y,
        n_splits=n_splits,
        random_state=random_state
    )
    class_to_index = meta["class_to_index"]

    X_use = prepare_vectors(X, metric)
    rows = []

    for i in range(len(ids)):
        p = P[i]

        k_eff = min(class_topk, len(p))
        top_class_idx = np.argpartition(-p, k_eff - 1)[:k_eff]
        top_class_idx = top_class_idx[np.argsort(-p[top_class_idx])]
        top_classes = classes[top_class_idx]

        cand = np.where(np.isin(y, top_classes))[0]
        cand = cand[cand != i]  # remove self

        if cand.size == 0:
            continue

        q = X_use[i]
        Xc = X_use[cand]

        scores = scores_for_metric(q, Xc, metric)

        cand_labels = y[cand]
        class_probs = np.array(
            [p[class_to_index[lab]] for lab in cand_labels],
            dtype=np.float32
        )

        # soft weighting by class confidence
        scores = scores * np.power(class_probs, beta)

        take = min(depth, len(cand))
        top_local = np.argpartition(-scores, take - 1)[:take]
        top_local = top_local[np.argsort(-scores[top_local])]
        nbrs = cand[top_local]

        rows.extend(build_run_rows_from_neighbors("soft_filtering", metric, i, nbrs, ids, y, depth))

    return pd.DataFrame(rows)

Cell 14 — RRF fusion function

In [14]:
def fuse_rrf(run_df, rrf_k=60):
    df = run_df.copy()

    # Safety check: Flat must not be included in the scalable hybrid fusion.
    if "flat" in df["method"].unique():
        raise ValueError("Flat retrieval is present in run_df. Remove it before hybrid fusion.")

    # RRF score
    df["rrf_score"] = 1.0 / (rrf_k + df["rank"].astype(float))

    fused = (
        df.groupby(
            ["query_id", "query_label", "doc_id", "doc_label"],
            as_index=False
        )["rrf_score"]
        .sum()
        .sort_values(["query_id", "rrf_score"], ascending=[True, False])
    )

    fused["rank"] = (
        fused.groupby("query_id")["rrf_score"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    fused["method"] = HYBRID_METHOD_NAME

    return fused[["method", "query_id", "query_label", "doc_id", "doc_label", "rank", "rrf_score"]]


Cell 15 — Evaluate Hybrid ranking

In [15]:
def evaluate_run(fused_df, topk_list):
    rows = []

    for qid, g in fused_df.groupby("query_id"):
        g = g.sort_values("rank")

        rels = (g["doc_label"].to_numpy() == g["query_label"].iloc[0]).astype(np.int32)
        total_rel = int(np.sum(rels))

        for K in topk_list:
            Ke = min(K, len(rels))

            rows.append({
                "query_id": qid,
                "query_label": int(g["query_label"].iloc[0]),
                "K": int(K),
                "precision@K": precision_at_k(rels, Ke),
                "recall@K": recall_at_k(rels, Ke, total_rel),
                "ndcg@K": ndcg_at_k(rels, Ke),
                "mrr@K": mrr_at_k(rels, Ke),
            })

    return pd.DataFrame(rows)

Cell 16 — Run Hybrid RRF without Flat for all embeddings and all similarity metrics

For each embedding and each similarity metric, the fused ranking is built from:

- classifier-guided retrieval
- IVF
- IVFPQ
- HNSW
- filtered static search

The outputs are:

- HybridRun
- PerQuery
- Summary


In [18]:
all_hybrid_runs = []
all_perquery = []
all_summary = []

for emb_name, path in EMBEDDING_FILES.items():
    print(f"=== Hybrid RRF without Flat for embedding: {emb_name} ===")

    if not os.path.exists(path):
        print("File not found:", path)
        continue

    _, X, y, ids = load_embedding_xlsx(path)

    for metric in SIM_METRICS:
        print(f"  -> metric: {metric}")
        t0 = time.time()

        run_parts = []

        # 1) classifier-guided / soft filtering
        run_parts.append(
            build_run_soft_filtering(
                X, y, ids, metric, FUSION_DEPTH,
                class_topk=CLASS_TOPK,
                beta=BETA,
                n_splits=N_SPLITS,
                random_state=RANDOM_STATE
            )
        )

        # 2) IVF
        run_parts.append(
            build_run_ivf(
                X, y, ids, metric, FUSION_DEPTH,
                nlist=IVF_NLIST,
                nprobe=IVF_NPROBE
            )
        )

        # 3) IVFPQ
        run_parts.append(
            build_run_ivfpq(
                X, y, ids, metric, FUSION_DEPTH,
                nlist=PQ_NLIST,
                nprobe=PQ_NPROBE,
                nbits=PQ_NBITS
            )
        )

        # 4) HNSW
        run_parts.append(
            build_run_hnsw(
                X, y, ids, metric, FUSION_DEPTH,
                M=HNSW_M,
                efc=HNSW_EF_CONSTRUCTION,
                efs=HNSW_EF_SEARCH
            )
        )

        # 5) filtered static (optional oracle-assisted setting)
        if INCLUDE_FILTERED_STATIC:
            run_parts.append(
                build_run_filtered_static(X, y, ids, metric, FUSION_DEPTH)
            )

        # merge all non-Flat runs
        runs = pd.concat(run_parts, ignore_index=True)
        runs.insert(0, "embedding", emb_name)

        # additional safety check
        assert "flat" not in runs["method"].unique(), "Flat must not be included in no-flat hybrid fusion."

        # fuse by RRF
        fused = fuse_rrf(runs, rrf_k=RRF_K)
        fused.insert(0, "embedding", emb_name)
        fused.insert(2, "metric", metric)
        all_hybrid_runs.append(fused)

        # evaluate
        perq = evaluate_run(fused, TOPK_LIST)
        perq.insert(0, "embedding", emb_name)
        perq.insert(1, "method", HYBRID_METHOD_NAME)
        perq.insert(2, "metric", metric)
        all_perquery.append(perq)

        # summary
        summary = (
            perq.groupby(["K"], as_index=False)
            .agg({
                "precision@K": "mean",
                "recall@K": "mean",
                "ndcg@K": "mean",
                "mrr@K": "mean"
            })
        )
        summary.insert(0, "embedding", emb_name)
        summary.insert(1, "method", HYBRID_METHOD_NAME)
        summary.insert(2, "metric", metric)
        all_summary.append(summary)

        print(f"     done in {round(time.time() - t0, 2)} sec")

df_hybrid_run = pd.concat(all_hybrid_runs, ignore_index=True)
df_perquery = pd.concat(all_perquery, ignore_index=True)
df_summary = pd.concat(all_summary, ignore_index=True)

print("Shapes:")
print("HybridRun:", df_hybrid_run.shape)
print("PerQuery :", df_perquery.shape)
print("Summary  :", df_summary.shape)

df_summary.head(20)


=== Hybrid RRF without Flat for embedding: bert_finetuned ===
E:\Experiments\Similarity Serach\Embeddings\bert_finetuned_embeddings.xlsx → embeddings shape: (2323, 768)
  -> metric: cosine
     done in 16.7 sec
  -> metric: dot
     done in 18.3 sec
  -> metric: neg_l2
     done in 25.15 sec
=== Hybrid RRF without Flat for embedding: gemini ===
E:\Experiments\Similarity Serach\Embeddings\Gemini_Embedding.xlsx → embeddings shape: (2323, 768)
  -> metric: cosine
     done in 17.23 sec
  -> metric: dot
     done in 18.92 sec
  -> metric: neg_l2
     done in 20.59 sec
=== Hybrid RRF without Flat for embedding: qwen3_8b ===
E:\Experiments\Similarity Serach\Embeddings\Qwen3_Embedding_8B.xlsx → embeddings shape: (2323, 4096)
  -> metric: cosine
     done in 109.9 sec
  -> metric: dot
     done in 105.09 sec
  -> metric: neg_l2
     done in 108.53 sec
=== Hybrid RRF without Flat for embedding: sbert ===
E:\Experiments\Similarity Serach\Embeddings\SBERT_Embedding_2_classification.xlsx → embeddi

,embedding,method,metric,K,precision@K,recall@K,ndcg@K,mrr@K
0,bert_finetuned,hybrid_rrf_no_flat,cosine,1,0.977615,0.015270,0.977615,0.977615
1,bert_finetuned,hybrid_rrf_no_flat,cosine,5,0.969178,0.075500,0.970948,0.979940
2,bert_finetuned,hybrid_rrf_no_flat,cosine,10,0.963840,0.150022,0.966622,0.980520
3,bert_finetuned,hybrid_rrf_no_flat,cosine,20,0.957856,0.297415,0.961418,0.980836
4,bert_finetuned,hybrid_rrf_no_flat,dot,1,0.977185,0.015242,0.977185,0.977185
5,bert_finetuned,hybrid_rrf_no_flat,dot,5,0.968833,0.075411,0.970523,0.979136
6,bert_finetuned,hybrid_rrf_no_flat,dot,10,0.962850,0.149688,0.965792,0.979799
7,bert_finetuned,hybrid_rrf_no_flat,dot,20,0.957835,0.297160,0.961211,0.980118
8,bert_finetuned,hybrid_rrf_no_flat,neg_l2,1,0.985794,0.015425,0.985794,0.985794
9,bert_finetuned,hybrid_rrf_no_flat,neg_l2,5,0.967198,0.075247,0.971230,0.987122


Cell 17 — Save Hybrid RRF without Flat results


In [ ]:
# save summary and per-query in Excel
with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as w:
    df_summary.to_excel(w, sheet_name="Summary", index=False)
    df_perquery.to_excel(w, sheet_name="PerQuery", index=False)

print("Saved summary:", OUT_XLSX)

# save large ranked list as CSV
df_hybrid_run.to_csv(OUT_CSV, index=False)

print("Saved ranked list:", OUT_CSV)


 Saved summary: hybrid_rrf_no_flat_summary.xlsx
 Saved ranked list: hybrid_rrf_no_flat_run.csv
